# 深度学习作业 HW03

姓名：邱宇
学号：20234080317

---

# 第2章：卷积神经网络

## 2.1 理论题

### 题目描述

假设输入张量形状为 3×32×32（通道数×高度×宽度），经过一个包含16个卷积核的卷积层，每个卷积核的形状为 3×5×5。填充（Padding）为2，步幅（Stride）为2。

1. 计算该卷积层输出的特征图（Feature Map）尺寸（通道数×高度×宽度）
2. 计算该卷积层的参数数量，并说明如何得出这个结果

第 1 题 输出特征图尺寸
已知条件：
输入图像：通道数 3，高 32，宽 32
卷积核数量：16 个，卷积核尺寸 5×5
填充 Padding=2，步幅 Stride=2
计算规则：
输出通道数 = 卷积核总个数
输出高 = (输入高 + 2 * 填充 - 卷积核尺寸) // 步幅 + 1
输出宽计算方式和输出高一致
计算过程：
输出通道数 = 16
输出高 = (32 + 2*2 - 5) // 2 + 1 = 31//2 + 1 = 15 + 1 = 16
输出宽 = 16
最终输出特征图尺寸：通道数 × 高 × 宽 = 16 × 16 × 16
第 2 题 单个像素乘法次数
单个输出通道单个像素的乘法次数 = 输入通道数 × 卷积核高 × 卷积核宽
计算：3 × 5 × 5 = 75
答案：75 次

## 2.2 编程题：实现 Max Pooling

In [1]:
import numpy as np
import torch
import torch.nn as nn

def max_pooling_manual(input_tensor, kernel_size, stride=1, padding=0):
    batch_size, channels, height, width = input_tensor.shape
    
    output_height = (height + 2 * padding - kernel_size) // stride + 1
    output_width = (width + 2 * padding - kernel_size) // stride + 1
    
    output = np.zeros((batch_size, channels, output_height, output_width))
    
    if padding > 0:
        padded_input = np.pad(input_tensor, 
                              ((0, 0), (0, 0), (padding, padding), (padding, padding)),
                              mode='constant')
    else:
        padded_input = input_tensor
    
    for b in range(batch_size):
        for c in range(channels):
            for i in range(output_height):
                for j in range(output_width):
                    h_start = i * stride
                    h_end = h_start + kernel_size
                    w_start = j * stride
                    w_end = w_start + kernel_size
                    
                    output[b, c, i, j] = np.max(padded_input[b, c, h_start:h_end, w_start:w_end])
    
    return output

class MaxPool2dManual(nn.Module):
    def __init__(self, kernel_size, stride=None, padding=0):
        super(MaxPool2dManual, self).__init__()
        self.kernel_size = kernel_size
        self.stride = stride if stride is not None else kernel_size
        self.padding = padding
    
    def forward(self, x):
        batch_size, channels, height, width = x.shape
        output_height = (height + 2 * self.padding - self.kernel_size) // self.stride + 1
        output_width = (width + 2 * self.padding - self.kernel_size) // self.stride + 1
        
        x_unfold = nn.functional.unfold(x, kernel_size=self.kernel_size, 
                                        padding=self.padding, stride=self.stride)
        x_unfold = x_unfold.view(batch_size, channels, self.kernel_size*self.kernel_size, -1)
        output = x_unfold.max(dim=2)[0]
        output = output.view(batch_size, channels, output_height, output_width)
        
        return output

print("=== Max Pooling 测试 ===")

batch_size = 2
channels = 3
height = 8
width = 8

torch.manual_seed(42)
test_tensor = torch.randn(batch_size, channels, height, width)

np_tensor = test_tensor.numpy()
manual_result_np = max_pooling_manual(np_tensor, kernel_size=2, stride=2, padding=0)

builtin_pool = nn.MaxPool2d(kernel_size=2, stride=2, padding=0)
builtin_result = builtin_pool(test_tensor)

manual_pool = MaxPool2dManual(kernel_size=2, stride=2, padding=0)
manual_result_torch = manual_pool(test_tensor)

print(f"输入形状: {test_tensor.shape}")
print(f"输出形状（手动numpy）: {manual_result_np.shape}")
print(f"输出形状（PyTorch内置）: {builtin_result.shape}")
print(f"输出形状（自定义PyTorch）: {manual_result_torch.shape}")

print(f"\n手动实现与内置函数结果差异: {torch.abs(builtin_result - torch.tensor(manual_result_np)).max().item():.6f}")
print(f"自定义实现与内置函数结果差异: {torch.abs(builtin_result - manual_result_torch).max().item():.6f}")
print("测试通过！" if torch.allclose(builtin_result, manual_result_torch) else "测试失败！")

=== Max Pooling 测试 ===
输入形状: torch.Size([2, 3, 8, 8])
输出形状（手动numpy）: (2, 3, 4, 4)
输出形状（PyTorch内置）: torch.Size([2, 3, 4, 4])
输出形状（自定义PyTorch）: torch.Size([2, 3, 4, 4])

手动实现与内置函数结果差异: 0.000000
自定义实现与内置函数结果差异: 0.000000
测试通过！


---

# 第3章：LeNet, AlexNet, VGG 和 NiN

## 3.1 理论题

### 题目描述

在VGG网络中，主要使用 3×3 卷积核来替代较大的卷积核（如 5×5 或 7×7）。假设输入和输出的通道数均为C。

1. 计算一个 5×5 卷积层的参数数量
2. 计算达到相同感受野所需的多个 3×3 卷积层的参数数量，并说明原因

约定：输入、输出通道数均为 C，卷积层无偏置
卷积层参数量 = 输入通道数 × 卷积核高 × 卷积核宽 × 输出通道数
第 1 题 单个 5×5 卷积层参数量
参数量 = C × 5 × 5 × C = 25 * C²
答案：25C²
第 2 题 两层串联 3×3 卷积层总参数量
单层 3×3 卷积参数量 = C × 3 × 3 × C = 9 * C²
两层总参数量 = 9C² + 9C² = 18C²
答案：18C²

## 3.2 编程题：实现 NiN Block

In [2]:
import torch
import torch.nn as nn

class NiNBlock(nn.Module):
    def __init__(self, in_channels, out_channels, kernel_size, stride=1, padding=0):
        super(NiNBlock, self).__init__()
        
        self.conv = nn.Conv2d(in_channels, out_channels, kernel_size, stride, padding)
        self.conv_1x1 = nn.Conv2d(out_channels, out_channels, kernel_size=1)
        self.relu = nn.ReLU(inplace=True)
    
    def forward(self, x):
        x = self.conv(x)
        x = self.relu(x)
        
        x = self.conv_1x1(x)
        x = self.relu(x)
        
        x = self.conv_1x1(x)
        x = self.relu(x)
        
        return x

print("=== NiN Block 测试 ===")

batch_size = 2
in_channels = 3
height = 32
width = 32

torch.manual_seed(42)
test_tensor = torch.randn(batch_size, in_channels, height, width)

nin_block = NiNBlock(in_channels=3, out_channels=16, kernel_size=3, stride=1, padding=1)
output = nin_block(test_tensor)

print(f"输入形状: {test_tensor.shape}")
print(f"输出形状: {output.shape}")
print(f"预期输出形状: ({batch_size}, 16, {height}, {width})")
print("测试通过！" if output.shape == (batch_size, 16, height, width) else "测试失败！")

=== NiN Block 测试 ===
输入形状: torch.Size([2, 3, 32, 32])
输出形状: torch.Size([2, 16, 32, 32])
预期输出形状: (2, 16, 32, 32)
测试通过！


---

# 第4章：Inception, Batch Normalization 和残差网络

## 4.1 理论题

### 题目描述

在小批量（Mini-batch）训练中，假设一批数据中有4个样本，其特征值分别为 x1=2, x2=4, x3=6, x4=8。

假设经过Batch Normalization层后，缩放参数 γ=2，偏移参数 β=1，且 ϵ=0。

计算这4个样本经过Batch Normalization后的输出值 y1, y2, y3, y4。

已知：
样本值：x1=2，x2=4，x3=6，x4=8
缩放参数 γ=2，平移参数 β=1，常数 ε=0
BN 计算步骤：
计算批次均值
均值 μ = (2 + 4 + 6 + 8) ÷ 4 = 5
计算批次方差
方差 σ² = [(2-5)² + (4-5)² + (6-5)² + (8-5)²] ÷ 4
= (9 + 1 + 1 + 9) ÷ 4
= 20 ÷ 4 = 5
归一化公式：x_hat = (x - μ) / √(σ²+ε)
因 ε=0，分母为√5
最终输出公式：y = γ * x_hat + β = 2*(x-5)/√5 + 1
逐个计算结果：
y1 = 1 - 6√5/5
y2 = 1 - 2√5/5
y3 = 1 + 2√5/5
y4 = 1 + 6√5/5

## 4.2 编程题：实现 Residual Block

In [3]:
import torch
import torch.nn as nn

class ResidualBlock(nn.Module):
    def __init__(self, in_channels, out_channels, use_1x1conv=False, stride=1):
        super(ResidualBlock, self).__init__()
        
        self.conv1 = nn.Conv2d(in_channels, out_channels, kernel_size=3, 
                               stride=stride, padding=1)
        self.bn1 = nn.BatchNorm2d(out_channels)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(out_channels, out_channels, kernel_size=3, 
                               stride=1, padding=1)
        self.bn2 = nn.BatchNorm2d(out_channels)
        
        self.use_1x1conv = use_1x1conv
        if use_1x1conv:
            self.conv3 = nn.Conv2d(in_channels, out_channels, kernel_size=1, 
                                   stride=stride)
            self.bn3 = nn.BatchNorm2d(out_channels)
    
    def forward(self, x):
        residual = x
        
        out = self.conv1(x)
        out = self.bn1(out)
        out = self.relu(out)
        
        out = self.conv2(out)
        out = self.bn2(out)
        
        if self.use_1x1conv:
            residual = self.conv3(x)
            residual = self.bn3(residual)
        
        out += residual
        out = self.relu(out)
        
        return out

print("=== Residual Block 测试 ===")

batch_size = 2
in_channels = 64
height = 32
width = 32

torch.manual_seed(42)
test_tensor = torch.randn(batch_size, in_channels, height, width)

residual_block1 = ResidualBlock(in_channels=64, out_channels=64, use_1x1conv=False)
output1 = residual_block1(test_tensor)
print(f"\n测试1 - 通道数不变:")
print(f"  输入形状: {test_tensor.shape}")
print(f"  输出形状: {output1.shape}")
print(f"  测试通过！" if output1.shape == test_tensor.shape else "  测试失败！")

residual_block2 = ResidualBlock(in_channels=64, out_channels=128, use_1x1conv=True, stride=2)
output2 = residual_block2(test_tensor)
print(f"\n测试2 - 通道数变化 + 下采样:")
print(f"  输入形状: {test_tensor.shape}")
print(f"  输出形状: {output2.shape}")
print(f"  预期形状: ({batch_size}, 128, {height//2}, {width//2})")
print(f"  测试通过！" if output2.shape == (batch_size, 128, height//2, width//2) else "  测试失败！")

=== Residual Block 测试 ===

测试1 - 通道数不变:
  输入形状: torch.Size([2, 64, 32, 32])
  输出形状: torch.Size([2, 64, 32, 32])
  测试通过！

测试2 - 通道数变化 + 下采样:
  输入形状: torch.Size([2, 64, 32, 32])
  输出形状: torch.Size([2, 128, 16, 16])
  预期形状: (2, 128, 16, 16)
  测试通过！


---

# 第5章：图像增广、微调与迁移学习

## 5.1 理论题

### 题目描述

在微调（Fine-tuning）任务中，假设我们有一个在ImageNet上预训练好的模型，现在要将其应用于自己的小数据集。

请说明微调的主要步骤和关键要点。

第 1 题 不同层设置不同学习率的原因
预训练网络的底层特征层，已经在大型数据集上学习到边缘、纹理、基础形状等通用视觉特征，这类特征适用范围广。如果使用大学习率更新参数，会破坏已训练好的有效特征，因此底层特征层使用较小学习率，或直接冻结参数。
网络顶层输出层是为预训练原始任务设计的，和新目标任务不匹配，参数需要重新学习来适配新任务，因此需要设置较大学习率，加快参数收敛。
第 2 题 数据集小且与源数据集相似的防过拟合策略
完全冻结主干特征提取网络，仅训练最后一层分类输出层，减少可训练参数，从源头抑制过拟合。
若精度不足，仅解冻网络最顶部少数几层，同时使用极小学习率微调，不改动底层通用特征。
搭配正则化手段：增加权重衰减、使用 Dropout 层；采用图像增广扩充训练样本。
训练时启用早停策略，验证集性能下降时立即停止训练；同时使用较小批次大小辅助降低过拟合风险。

## 5.2 编程题：实现图像增广 Pipeline

In [4]:
import torch
from torchvision import transforms
import numpy as np

def create_augmentation_pipeline():
    pipeline = transforms.Compose([
        transforms.RandomResizedCrop(224, scale=(0.08, 1.0)),
        transforms.RandomHorizontalFlip(p=0.5),
        transforms.ColorJitter(
            brightness=0.5,
            contrast=0.5,
            saturation=0.5
        ),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225]
        )
    ])
    
    return pipeline

print("=== 图像增广 Pipeline 测试 ===")

test_image = np.random.randint(0, 256, (256, 256, 3), dtype=np.uint8)

from PIL import Image
pil_image = Image.fromarray(test_image)

print(f"原始图像形状: {test_image.shape}")
print(f"PIL图像模式: {pil_image.mode}")

augmentation = create_augmentation_pipeline()

print("\n应用增广后的张量形状:")
for i in range(3):
    augmented_tensor = augmentation(pil_image)
    print(f"  第{i+1}次增广: {augmented_tensor.shape}")

print("\n增广流水线组成:")
for i, transform in enumerate(augmentation.transforms):
    print(f"  {i+1}. {transform}")

print("\n测试通过！")

=== 图像增广 Pipeline 测试 ===
原始图像形状: (256, 256, 3)
PIL图像模式: RGB

应用增广后的张量形状:
  第1次增广: torch.Size([3, 224, 224])
  第2次增广: torch.Size([3, 224, 224])
  第3次增广: torch.Size([3, 224, 224])

增广流水线组成:
  1. RandomResizedCrop(size=(224, 224), scale=(0.08, 1.0), ratio=(0.75, 1.3333), interpolation=bilinear, antialias=True)
  2. RandomHorizontalFlip(p=0.5)
  3. ColorJitter(brightness=(0.5, 1.5), contrast=(0.5, 1.5), saturation=(0.5, 1.5), hue=None)
  4. ToTensor()
  5. Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])

测试通过！


---

# 第6章：目标检测与标签平滑

## 6.1 理论题

### 题目描述

在目标检测中，IoU（Intersection over Union）是衡量预测框与真实框重叠程度的指标。

给定：
- 真实框（Ground Truth）A = [10, 10, 50, 50] （格式：[左上角x, 左上角y, 右下角x, 右下角y]）
- 预测框（Prediction Box）B = [30, 30, 70, 70]

计算这两个框的IoU值。

框格式说明：[左上角 x, 左上角 y, 右下角 x, 右下角 y]
真实框 A = [10,10,50,50]
预测框 B = [30,30,70,70]
IoU = 交集面积 / 并集面积
计算步骤：
求交集坐标
交集左上角 x = max (10, 30) = 30
交集左上角 y = max (10, 30) = 30
交集右下角 x = min (50, 70) = 50
交集右下角 y = min (50, 50) = 50
交集尺寸与面积
交集宽 = 50 - 30 = 20
交集高 = 50 - 30 = 20
交集面积 = 20 × 20 = 400
单个边框面积
框 A 面积 = (50-10) × (50-10) = 1600
框 B 面积 = (70-30) × (70-30) = 1600
并集面积
并集面积 = 1600 + 1600 - 400 = 2800
计算 IoU
IoU = 400 / 2800 = 1/7
近似小数：1/7 ≈ 0.1429
最终答案：IoU = 1/7


## 6.2 编程题：实现标签平滑的交叉熵损失

In [5]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class LabelSmoothingCrossEntropy(nn.Module):
    def __init__(self, num_classes, epsilon=0.1):
        super(LabelSmoothingCrossEntropy, self).__init__()
        self.num_classes = num_classes
        self.epsilon = epsilon
    
    def forward(self, logits, targets):
        if targets.dim() == 1:
            targets_onehot = F.one_hot(targets, num_classes=self.num_classes).float()
        else:
            targets_onehot = targets
        
        smooth_targets = (1 - self.epsilon) * targets_onehot + self.epsilon / self.num_classes
        log_probs = F.log_softmax(logits, dim=1)
        loss = -torch.sum(smooth_targets * log_probs, dim=1).mean()
        
        return loss

print("=== 标签平滑交叉熵损失测试 ===")

num_classes = 10
epsilon = 0.1
batch_size = 4

torch.manual_seed(42)
logits = torch.randn(batch_size, num_classes)
targets = torch.tensor([0, 2, 5, 7])

loss_fn = LabelSmoothingCrossEntropy(num_classes=num_classes, epsilon=epsilon)
loss = loss_fn(logits, targets)

print(f"输入 logits 形状: {logits.shape}")
print(f"目标标签: {targets}")
print(f"类别数 K: {num_classes}")
print(f"平滑因子 ϵ: {epsilon}")
print(f"\n计算的损失值: {loss.item():.4f}")

standard_ce = F.cross_entropy(logits, targets)
print(f"标准交叉熵损失: {standard_ce.item():.4f}")

print("\n标签平滑效果说明:")
print(f"  - 真实类别标签: 从 1 -> {1-epsilon}")
print(f"  - 其他类别标签: 从 0 -> {epsilon/(num_classes):.4f}")
print(f"  - 作用: 防止模型过度自信，提高泛化能力")

=== 标签平滑交叉熵损失测试 ===
输入 logits 形状: torch.Size([4, 10])
目标标签: tensor([0, 2, 5, 7])
类别数 K: 10
平滑因子 ϵ: 0.1

计算的损失值: 2.6996
标准交叉熵损失: 2.6813

标签平滑效果说明:
  - 真实类别标签: 从 1 -> 0.9
  - 其他类别标签: 从 0 -> 0.0100
  - 作用: 防止模型过度自信，提高泛化能力
